In [2]:
#enahcned
import xt_fns as xf
import numpy as np
import matplotlib.pyplot as plt
import batman
import glob

#Stellar parameters
Rs = 0.805 #radius (Rsun)
Ms = 0.846 #mass (Msun)
T = 0.34 #kT of corona in keV

#planetary parameters
Rp_surf = 8135789600.0

for Z in [1,5,10]:
    # file_pattern = '../batmanX-rays-main/tau_profiles/tau_HD189_*eV_'+str(Z)+'Z.npy'
    # big_files = sorted(glob.glob(file_pattern))
    # big_files = big_files.append('../batmanX-rays-main/tau_profiles/tau_HD189_*eV_'+str(Z)+'Z_x4.npy')
    profiles_array = np.load(f'../batmanX-rays-main/tau_profiles/tau_HD189_{Z}Z_all.npy', allow_pickle=True)
    profiles_array_shock = np.load(f'../batmanX-rays-main/tau_profiles/tau_HD189_{Z}Z_x4.npy', allow_pickle=True)

    out = []
    for profile,profile_shock in zip(profiles_array, profiles_array_shock):
        E = profile['E']
        print(f'\rCalculating transit for {E} eV, {Z} Zsun', end='')
        x = profile['rs']
        tau_array = profile['taus']
        x2 = profile_shock['rs']
        tau_array_bow = profile_shock['taus']

        wavelength = E
        # x2 = x
        # tau_array_bow = tau_array

        Rmax = max(x)/Rp_surf
        Rp = Rp_surf*Rmax #planet radius in cm 
        RpRs = Rp/(Rs*6.957e10) #Rmax/Rs
        aRs = 8.2830 #a/Rs
        # inc =  #inclination
        inc = 85.580 #inclination
        t0 = 1 #phase of transit centre


        #get emission scale height in stellar radii
        He = xf.HfromT(T, Rs, Ms)
        #get emission scale height in batman coordinates
        he = xf.hfromH(He)
        Nscale = 6
        #photosphere edge in batman coords
        Rx = xf.calcRx(he, Nscale)
        intVals = xf.get_intVals(he, Nscale)

        #set up the object to store the transit parameters
        params = batman.TransitParams()
        params.t0 = t0               #time of inferior conjunction
        params.per = 1               #orbital period - set to 1 to keep within phase definitions
        params.rp = RpRs * Rx        #planet radius (in units of stellar radii)
        params.a = aRs * Rx          #semi-major axis (in units of stellar radii)
        params.inc = inc             #orbital inclination (in degrees)
        params.ecc = 0.              #eccentricity
        params.w = 90.               #longitude of periastron (in degrees)
        params.limb_dark = "custom"  #limb darkening model
        params.u = [0]*6             #limb darkening coefficients
        params.u[0] = intVals
        params.tau = tau_array
        params.tau2 = tau_array_bow

        #initialise a batman model at the calcuation phase times
        #Setting up times for model
        phaSt, phaFi = 0.9, 1.1 #phases to model
        numBins = 500 #500 #number of bins to calculate the light curve at
        binPhases = np.linspace(phaSt, phaFi, numBins) #bin centres

        #set up batman model
        mod = batman.TransitModel(params, binPhases, fac=5e-4)
        #get the fluxes
        flux1 = mod.light_curve(params)

        out.append({'E': E, 'binPhases': binPhases, 'transmission_frac': flux1})

        # result = np.column_stack((np.full_like(binPhases, wavelength), binPhases, flux1))
        # out.append(result)

        # out = np.append(out, np.column_stack((binPhases, flux1)),axis=1)
        # np.save(f'../copy/transit_profiles/HD189_{wavelength}_{Z}Z.npy', np.column_stack((binPhases, flux1)))
    np.save(f'../copy/transit_profiles/HD189_{Z}Z_shock_x4.npy', out,allow_pickle=True)

    # np.savetxt(f'../copy/transit_profiles/HD189_{Z}Z_all.csv', np.vstack(out),delimiter= ',',header='energy,bin,Phase,flux')

Calculating transit for 1999.6020834059173 eV, 1 Zsun

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [5]:
Z = 1
profiles_array_shock = np.load(f'../batman_xrays_atmo_escape/tau_profiles/tau_HD189_{Z}Z_shock_enh1.npy', allow_pickle=True)
# print(np.shape(profiles_array_shock))
for pf in profiles_array_shock:
    print(pf)

{'E': 166.0849835407508, 'rs': array([0.        , 0.05308711, 0.10617422, 0.15926133, 0.21234844,
       0.26543556, 0.31852267, 0.37160978, 0.42469689, 0.477784  ,
       0.53087111, 0.58395822, 0.63704533, 0.69013244, 0.74321956,
       0.79630667, 0.84939378, 0.90248089, 0.955568  , 1.00865511,
       1.06174222, 1.11482933, 1.16791645, 1.22100356, 1.27409067,
       1.32717778, 1.38026489, 1.433352  , 1.48643911, 1.53952622,
       1.59261333, 1.64570045, 1.69878756, 1.75187467, 1.80496178,
       1.85804889, 1.911136  , 1.96422311, 2.01731022, 2.07039733,
       2.12348445, 2.17657156, 2.22965867, 2.28274578, 2.33583289,
       2.38892   , 2.44200711, 2.49509422, 2.54818134, 2.60126845,
       2.65435556, 2.70744267, 2.76052978, 2.81361689, 2.866704  ,
       2.91979111, 2.97287822, 3.02596534, 3.07905245, 3.13213956,
       3.18522667, 3.23831378, 3.29140089, 3.344488  , 3.39757511,
       3.45066222, 3.50374934, 3.55683645, 3.60992356, 3.66301067,
       3.71609778, 3.76918489, 

In [ ]:
data = np.genfromtxt(f'../batman_xrays_atmo_escape/tau_profiles/tau_HD189_{Z}Z_shock_enh1.csv', delimiter=',')
E = data[:,0]
binPhases = data[:,1:101]
flux1 = data[:,101:]


[[2.11804927e+02 2.11804927e+02 2.11804927e+02 ... 6.55472214e-09
  6.55472214e-09 6.55472214e-09]
 [2.11118260e+02 2.11118260e+02 2.11118260e+02 ... 6.55372750e-09
  6.55372750e-09 6.55372750e-09]
 [2.10435246e+02 2.10435246e+02 2.10435246e+02 ... 6.55273697e-09
  6.55273697e-09 6.55273697e-09]
 ...
 [1.00000000e+01 1.00000000e+01 1.00000000e+01 ... 1.55478270e-11
  1.55478270e-11 1.55478270e-11]
 [1.00000000e+01 1.00000000e+01 1.00000000e+01 ... 1.54702329e-11
  1.54702329e-11 1.54702329e-11]
 [1.00000000e+01 1.00000000e+01 1.00000000e+01 ... 1.53930156e-11
  1.53930156e-11 1.53930156e-11]]
